# Explore an overnight Garmin recording

Load a local FIT file, inspect acceleration and numbered BBI data, select a diary interval, and analyze movement-associated HR responses and quiet-period HRV. Configure inputs in `paths.local.json` as described in the repository README. Personal recordings and notebook outputs are not distributed.

In [ ]:
import importlib.util
import subprocess
import sys

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])

ensure_package("fitparse")

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from fitparse import FitFile

import matplotlib.pyplot as plt
import matplotlib as mpl 
import seaborn as sns

# Use the notebook backend; no desktop Qt installation is required.
plt.style.use("seaborn-v0_8-whitegrid")  # Use seaborn style for better aesthetics

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

import sys
PROCESSING = Path.cwd() if (Path.cwd() / "notebook_paths.py").exists() else Path.cwd() / "offline_processing"
if str(PROCESSING.resolve()) not in sys.path:
    sys.path.insert(0, str(PROCESSING.resolve()))
from notebook_paths import input_path

FIT_PATH = input_path("fit_path")

assert FIT_PATH.exists(), f"FIT file not found: {FIT_PATH.resolve()}"
FIT_PATH.resolve()

## Read The FIT File

In [ ]:
fit = FitFile(str(FIT_PATH))
messages = list(fit.get_messages())

message_counts = Counter(message.name for message in messages)
message_counts_df = (
    pd.DataFrame(message_counts.most_common(), columns=["message", "count"])
    .sort_values(["count", "message"], ascending=[False, True])
    .reset_index(drop=True)
)

print(f"Loaded {FIT_PATH.name}")
print(f"Total messages: {len(messages):,}")
message_counts_df.head(30)

In [ ]:
fields_by_message = defaultdict(Counter)
for message in messages:
    for field in message:
        fields_by_message[message.name][field.name] += 1

for message_name in ["record", "accelerometer_data", "session", "activity"]:
    print(f"\n{message_name} fields")
    if message_name not in fields_by_message:
        print("  not present")
        continue
    for field_name, count in fields_by_message[message_name].most_common():
        print(f"  {field_name}: {count}")

## Convert FIT Messages To DataFrames

In [ ]:
def message_to_dict(message):
    row = {}
    for field in message:
        name = field.name
        value = field.value
        if name in row:
            if not isinstance(row[name], list):
                row[name] = [row[name]]
            row[name].append(value)
        else:
            row[name] = value
    return row

def messages_to_dataframe(message_name):
    rows = [message_to_dict(message) for message in messages if message.name == message_name]
    return pd.DataFrame(rows)

record_df = messages_to_dataframe("record")
session_df = messages_to_dataframe("session")
activity_df = messages_to_dataframe("activity")

print(f"record_df shape: {record_df.shape}")
print(f"session_df shape: {session_df.shape}")
print(f"activity_df shape: {activity_df.shape}")
record_df.head()

## BBI Developer Fields

Current schema 3 stores numbered intervals in `bbi_history` with callback-arrival offsets in `bbi_rx_ms`. Scalar fields such as `bbi_count`, `bbi_latest`, and `bbi_total` are diagnostics. The following inspection cell also tolerates the legacy `bbi_1` through `bbi_8` fields; individual intervals for HRV are decoded later with `fit_bbi`.

In [ ]:
bbi_columns = [
    "timestamp",
    "heart_rate",
    "bbi_count",
    "bbi_latest",
    "bbi_total",
    "bbi_1",
    "bbi_2",
    "bbi_3",
    "bbi_4",
    "bbi_5",
    "bbi_6",
    "bbi_7",
    "bbi_8",
    "accel_samples",
]
available_bbi_columns = [column for column in bbi_columns if column in record_df.columns]
bbi_df = record_df[available_bbi_columns].copy()

if "timestamp" in bbi_df.columns:
    bbi_df = bbi_df.sort_values("timestamp")

LOCAL_TZ = "Europe/Rome"

local_times = pd.to_datetime(
    bbi_df["timestamp"], utc=True
).dt.tz_convert(LOCAL_TZ)

bbi_df["timestamp"] = local_times
bbi_df["timestamp"] = bbi_df["timestamp"].dt.tz_localize(None)

print(bbi_df.tail(20).to_string(index=False))

In [ ]:
plt.figure(figsize=(12, 6))

bbi_df_plot = bbi_df.copy()
bbi_df_plot["timestamp"] = bbi_df_plot["timestamp"].dt.tz_localize(None)

plt.plot(bbi_df["timestamp"], bbi_df["heart_rate"], marker="o", linestyle="-")
plt.xlabel("Time")
plt.ylabel("Heart Rate")
plt.title("Heart Rate Data Over Time")
plt.legend()
plt.show()

In [ ]:
if not bbi_df.empty and "timestamp" in bbi_df.columns:
    duration_s = (bbi_df["timestamp"].max() - bbi_df["timestamp"].min()).total_seconds()
else:
    duration_s = None

nonzero_bbi_rows = bbi_df[bbi_df.get("bbi_count", pd.Series(dtype="int64")).fillna(0) > 0]

print(f"Record rows: {len(bbi_df):,}")
if duration_s is not None:
    print(f"Record duration: {duration_s:.1f} seconds")
print(f"Rows with BBI intervals: {len(nonzero_bbi_rows):,}")

if "bbi_total" in bbi_df.columns:
    print(f"Final bbi_total: {bbi_df['bbi_total'].max()}")

if nonzero_bbi_rows.empty:
    print("No BBI intervals were recorded in this file. Heart rate is present, but heartBeatIntervals were not available during this test run.")
else:
    print(nonzero_bbi_rows.head(20).to_string(index=False))

## Expand Accelerometer Batches

`accelerometer_data` messages store x/y/z samples as arrays. This cell expands them into one row per accelerometer sample and reconstructs a sample timestamp from `timestamp`, `timestamp_ms`, and `sample_time_offset`.

In [ ]:
def flatten_samples(value):
    if value is None:
        return []
    if isinstance(value, tuple):
        return list(value)
    if isinstance(value, list):
        flattened = []
        for item in value:
            if isinstance(item, (tuple, list)):
                flattened.extend(item)
            else:
                flattened.append(item)
        return flattened
    return [value]

def expand_accelerometer_messages():
    rows = []
    sample_index = 0

    for message_index, message in enumerate(message for message in messages if message.name == "accelerometer_data"):
        row = message_to_dict(message)
        xs = flatten_samples(row.get("calibrated_accel_x"))
        ys = flatten_samples(row.get("calibrated_accel_y"))
        zs = flatten_samples(row.get("calibrated_accel_z"))
        offsets = flatten_samples(row.get("sample_time_offset"))

        n = min(len(xs), len(ys), len(zs))
        timestamp = row.get("timestamp")
        timestamp_ms = row.get("timestamp_ms") or 0

        base_time = pd.NaT
        if timestamp is not None:
            base_time = pd.Timestamp(timestamp) + pd.to_timedelta(timestamp_ms, unit="ms")

        for i in range(n):
            offset_ms = offsets[i] if i < len(offsets) else None
            sample_time = pd.NaT
            if pd.notna(base_time):
                sample_time = base_time
                if offset_ms is not None:
                    sample_time = sample_time + pd.to_timedelta(offset_ms, unit="ms")

            rows.append({
                "sample_index": sample_index,
                "message_index": message_index,
                "sample_time": sample_time,
                "offset_ms": offset_ms,
                "accel_x": xs[i],
                "accel_y": ys[i],
                "accel_z": zs[i],
            })
            sample_index += 1

    return pd.DataFrame(rows)

accel_df = expand_accelerometer_messages()

LOCAL_TZ = "Europe/Rome"

local_times = pd.to_datetime(
    accel_df["sample_time"], utc=True
).dt.tz_convert(LOCAL_TZ)

accel_df["sample_time"] = local_times
accel_df["sample_time"] = accel_df["sample_time"].dt.tz_localize(None)


# Keep the full recording and UTC clock so diary-cell reruns can expand the crop.
accel_df_full = accel_df
accel_times_utc_full = pd.DatetimeIndex(local_times).tz_convert("UTC")
accel_full_fit_path = FIT_PATH.resolve()

print("Recording start:", local_times.min())
print("Recording end:  ", local_times.max())

print(f"accel_df shape: {accel_df.shape}")
accel_df.head()

In [ ]:
if not accel_df.empty:
    duration_s = (accel_df["sample_time"].max() - accel_df["sample_time"].min()).total_seconds()
    sample_rate_estimate = len(accel_df) / duration_s if duration_s else None

    print(f"Accelerometer samples: {len(accel_df):,}")
    print(f"Accelerometer time range: {accel_df['sample_time'].min()} to {accel_df['sample_time'].max()}")
    print(f"Accelerometer duration: {duration_s:.1f} seconds")
    if sample_rate_estimate is not None:
        print(f"Estimated sample rate: {sample_rate_estimate:.2f} Hz")

    print("\nAcceleration summary:")
    print(accel_df[["accel_x", "accel_y", "accel_z"]].describe().to_string())
else:
    print("No accelerometer_data messages were found.")

In [ ]:
sns.set_context("notebook", font_scale=1.5)
accel_df_plot = accel_df.copy()
accel_df_plot["sample_time"] = accel_df_plot["sample_time"].dt.tz_localize(None)

plt.figure(figsize=(12, 6))
plt.plot(accel_df_plot["sample_time"], accel_df_plot["accel_x"], label="Acc X")
plt.plot(accel_df_plot["sample_time"], accel_df_plot["accel_y"], label="Acc Y")
plt.plot(accel_df_plot["sample_time"], accel_df_plot["accel_z"], label="Acc Z")
plt.xlabel("Time")
plt.ylabel("Acceleration")
plt.title("Accelerometer Data Over Time")
plt.legend(loc = "upper right")
plt.show()

## Plot accelerometer together with BBIs

In [ ]:
acc_norm = (accel_df["accel_x"]**2 + accel_df["accel_y"]**2 + accel_df["accel_z"]**2)**0.5
f, ax = plt.subplots(2, 1,figsize=(12, 6), sharex = True)
ax[0].plot(accel_df["sample_time"], acc_norm, label="Acceleration Norm", color="black")
ax[0].set_xlabel("Time")
ax[0].set_ylabel("Acceleration (m/s^2)")
ax[1].plot(accel_df["sample_time"], accel_df["accel_x"], label="Acc X", color="red")
ax[1].plot(accel_df["sample_time"], accel_df["accel_y"], label="Acc Y", color="green")
ax[1].plot(accel_df["sample_time"], accel_df["accel_z"], label="Acc Z", color="blue")
ax[1].set_xlabel("Time")
ax[1].set_ylabel("Acceleration (m/s^2)")
plt.show()

In [ ]:
f, ax = plt.subplots(2, 1,figsize=(12, 6), sharex = True)
ax[0].plot(accel_df_plot["sample_time"], accel_df_plot["accel_x"], label="Acc X")
ax[0].plot(accel_df_plot["sample_time"], accel_df_plot["accel_y"], label="Acc Y")
ax[0].plot(accel_df_plot["sample_time"], accel_df_plot["accel_z"], label="Acc Z")
ax[0].set_xlabel("Time")
ax[0].set_ylabel("Acceleration")
ax[0].set_title("Accelerometer Data Over Time")

ax[1].plot(bbi_df_plot["timestamp"], bbi_df_plot["heart_rate"], marker="o", linestyle="-", color="blue")
ax[1].set_xlabel("Time")
ax[1].set_ylabel("Heart Rate")
ax[1].set_title("Heart Rate Data Over Time")

plt.tight_layout()
plt.show()

## Load sleep diary and cut the signal

`Date` is the calendar date of **lightsoff**, even when lights off is after midnight.
Wakeup is on the next calendar day when its clock time is earlier than lightsoff.
The matching row is selected by overlap with the full recording, in `LOCAL_TZ`,
not by row number or the date in the filename. Missing or multiple matches raise
an error; incomplete diary rows are listed. The workbook is read-only.

The crop is **[lightsoff, wakeup)**, restricted to recorded data. The original
acceleration and UTC timestamps are retained by **Expand Accelerometer Batches**,
so rerunning this cell after a diary edit does not progressively shorten the data.
Rerun the burst and subsequent analysis cells after changing the diary.


In [ ]:
import importlib
import sys
from IPython.display import display

processing_dir = Path.cwd()
if not (processing_dir / "sleep_diary.py").exists():
    processing_dir = processing_dir / "offline_processing"
if not (processing_dir / "sleep_diary.py").exists():
    raise FileNotFoundError("Run from the repository root or offline_processing folder.")
if str(processing_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(processing_dir.resolve()))
import sleep_diary
importlib.reload(sleep_diary)

if "accel_df_full" not in globals() or accel_full_fit_path != FIT_PATH.resolve():
    raise RuntimeError("Rerun Expand Accelerometer Batches for the current FIT_PATH first.")
if accel_times_utc_full.empty or accel_times_utc_full.hasnans:
    raise ValueError("A nonempty acceleration stream with valid timestamps is required.")

ensure_package("openpyxl")
DIARY_PATH = input_path("diary_path")
diary = pd.read_excel(DIARY_PATH)
sample_steps = pd.Series(accel_times_utc_full).diff()
sample_period = sample_steps[sample_steps > pd.Timedelta(0)].median()
if pd.isna(sample_period):
    raise ValueError("Cannot determine the sample period from the recording.")
diary_row, diary_skipped = sleep_diary.select_sleep_interval(
    diary, accel_times_utc_full.min(), accel_times_utc_full.max() + sample_period,
    timezone=LOCAL_TZ,
)
start_bedtime = diary_row["lightsoff_at"]
end_bedtime = diary_row["wakeup_at"]
sleep_start_utc = diary_row["recorded_start"].tz_convert("UTC")
sleep_end_utc = diary_row["recorded_end"].tz_convert("UTC")
sleep_mask = (accel_times_utc_full >= sleep_start_utc) & (accel_times_utc_full < sleep_end_utc)
accel_df = accel_df_full.loc[sleep_mask].copy()
if accel_df.empty:
    raise ValueError("The selected diary interval contains no accelerometer samples.")

display(diary_row[[
    "excel_row", "Date", "lightsoff_at", "wakeup_at",
    "recorded_start", "recorded_end", "full_diary_interval_recorded",
]].rename("Matched sleep diary entry"))
if not diary_skipped.empty:
    display(diary_skipped)
print(f"Kept {len(accel_df):,} of {len(accel_df_full):,} acceleration samples.")
print(f"Analysis interval: {sleep_start_utc.tz_convert(LOCAL_TZ)} to {sleep_end_utc.tz_convert(LOCAL_TZ)} (end exclusive)")
if not diary_row["full_diary_interval_recorded"]:
    print("Caution: this recording covers only part of the diary interval.")


## Wrist Acceleration Bursts (40 mg exploratory setting)

Run after **Expand Accelerometer Batches**. This uses the current exploratory wrist threshold of **0.040 g** on the upper-minus-lower envelope of the band-pass-filtered vector magnitude, not on an individual axis or raw magnitude. The detector applies the 0.1-10 Hz, order-8 Butterworth filter once, retains groups of ten extrema, and merges gaps strictly shorter than five seconds.

**Units matter:** the September overnight FIT has values near 1,000 at rest despite its calibrated-field metadata saying "g". The cell explicitly assumes **mg** and divides by 1,000, then checks that median magnitude is near 1 g. This is an inference from the recorded scale, not a calibration; use a stationary recording to confirm it. Change `ACCEL_INPUT_UNIT` for files truly stored in g.

The cell sorts timestamps, averages duplicate magnitudes, and interpolates onto a 100 Hz grid because native FIT timestamps are jittered. It rejects missing values and gaps over 0.25 s rather than bridging unobserved movement. This resampling and the corrected envelope alignment can change detections relative to the old implementation: inspect the plots before treating the supplied threshold as validated on this Garmin pipeline. Edge transients remain possible.

Results are in `bursts_df`: start/end (end exclusive), duration, filtered peak-to-peak amplitude in g, and AUC in **g*s** (previous code used sample units). `burst_signals` holds full-resolution filtered magnitude and envelope. The overview plots one-second envelope maxima only for display; detection uses every regular-grid sample. Set `PLOT_START` to a timestamp in the same timezone as `accel_df` to inspect a different window.

In [ ]:
import importlib
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

processing_dir = Path.cwd()
if not (processing_dir / "detect_acc_bursts.py").exists():
    processing_dir = processing_dir / "offline_processing"
if not (processing_dir / "detect_acc_bursts.py").exists():
    raise FileNotFoundError("Run from the repository root or offline_processing folder.")
if str(processing_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(processing_dir.resolve()))
import detect_acc_bursts as acc_bursts
importlib.reload(acc_bursts)

ACCEL_INPUT_UNIT = "mg"  # Explicit scale for this overnight FIT, despite its field labels.
SAMPLING_RATE_HZ = 100.0  # Native FIT stream, not the app's 25 Hz callback.
WRIST_THRESHOLD_G = 0.040
PLOT_START = None  # E.g. "2026-09-13 02:00:00", using the FIT timestamp timezone.
PLOT_MINUTES = 5

acc_magnitude_g, acc_quality = acc_bursts.prepare_acceleration(
    accel_df, sampling_rate=SAMPLING_RATE_HZ, input_unit=ACCEL_INPUT_UNIT,
    max_gap_s=0.25,
)
display(pd.Series(acc_quality, name="Acceleration input checks"))
if not 0.5 <= acc_quality["median_magnitude_g"] <= 1.5:
    raise ValueError("Median magnitude is not near 1 g. Check ACCEL_INPUT_UNIT and calibration.")

bursts_df, burst_signals = acc_bursts.detect_bursts(
    acc_magnitude_g,
    sampling_rate=SAMPLING_RATE_HZ,
    envelope=True,
    resample_envelope=True,
    alfa=WRIST_THRESHOLD_G,
    merge_gap_s=5.0,
    return_signals=True,
)
recording_s = (acc_magnitude_g.index[-1] - acc_magnitude_g.index[0]).total_seconds() + 1 / SAMPLING_RATE_HZ
burst_s = bursts_df["duration"].dt.total_seconds().sum()
print(f"{len(bursts_df):,} bursts in {recording_s / 3600:.2f} hours at {WRIST_THRESHOLD_G * 1000:.0f} mg")
print(f"Merged burst duration: {burst_s / 60:.1f} min ({100 * burst_s / recording_s:.1f}% of recording; includes merged gaps)")
display(bursts_df.assign(
    duration_s=bursts_df["duration"].dt.total_seconds(),
    peak_to_peak_mg=bursts_df["peak-to-peak"] * 1000,
).head(20))

window_start = acc_magnitude_g.index[0] if PLOT_START is None else pd.Timestamp(PLOT_START)
window_end = min(window_start + pd.Timedelta(minutes=PLOT_MINUTES), acc_magnitude_g.index[-1])
filtered_window = burst_signals["filtered"].loc[window_start:window_end]
envelope_window = burst_signals["score"].loc[window_start:window_end]
if filtered_window.empty:
    raise ValueError("PLOT_START selects no samples; choose a timestamp inside the recording.")

fig, axes = plt.subplots(2, 1, figsize=(14, 7), constrained_layout=True)
overview = burst_signals["score"].resample("1s").max() * 1000
axes[0].plot(overview.index, overview, color="teal", linewidth=0.7, label="Envelope: 1 s maximum")
axes[0].set_title("Wrist bursts: full recording")
axes[0].set_yscale("symlog", linthresh=WRIST_THRESHOLD_G * 1000)
axes[1].plot(filtered_window.index, filtered_window * 1000, color="0.5", linewidth=0.6, label="Filtered magnitude")
axes[1].plot(envelope_window.index, envelope_window * 1000, color="teal", linewidth=1, label="Envelope")
axes[1].set_title(f"Detail: {window_start} to {window_end}")
for ax in axes:
    ax.axhline(WRIST_THRESHOLD_G * 1000, color="firebrick", linestyle="--", label=f"{WRIST_THRESHOLD_G * 1000:g} mg threshold")
    ax.set_ylabel("Acceleration (mg)")
    ax.set_xlabel("Time (FIT timestamps)")
    ax.legend(loc="upper right")
axes[0].set_ylabel("Envelope (mg; symlog scale)")
for burst in bursts_df.itertuples(index=False):
    axes[0].axvspan(burst.start, burst.end, color="red", alpha=0.25)
    if burst.end > window_start and burst.start < window_end:
        axes[1].axvspan(max(burst.start, window_start), min(burst.end, window_end), color="red", alpha=0.25)
axes[1].set_xlim(filtered_window.index[0], filtered_window.index[-1])
plt.show()

## Recorded HR Responses by Burst AUC

Run after the wrist-burst cell. **This section now uses the recorded HR values in `bbi_df["heart_rate"]` (or `hr_bpm` if already renamed), not the decoded BBI-history stream.** Despite its name, `bbi_df` is a FIT record table. No 60,000 / BBI conversion or callback averaging is used here. The sensor source and upstream HR smoothing are not established by this table.

The analysis adapts the [paper's HR methods](https://www.nature.com/articles/s41598-025-29723-7). The current implementation samples HR at 1 Hz from -20 through +49 s, with 15 baseline samples in [-20, -5), and searches for the post-onset maximum from +1 through +49 s. These settings differ from the paper's -19 through +40 s epoch and ten-sample baseline [-19, -9). Normalization is 100 * (HR / baseline - 1). **Thirty-second movement isolation** is configured through `ISOLATION_SECONDS`. `EXCLUDE_LATE_OVERLAP=True` additionally excludes another movement starting anywhere through +49 s, protecting the full sampled response window.

**Input handling and limits:**
- `bbi_hr_input` is an analysis copy made from `bbi_df`; the original table is not renamed, reindexed, or modified. Invalid HR rows stay missing. Configurable HR bounds are screening, not validated artifact correction.
- The earlier cells convert timestamps to local wall time and remove timezone labels. Here, naive HR and acceleration/burst times are explicitly localized with `NAIVE_INPUT_TIMEZONE = LOCAL_TZ` and converted to UTC together. Already timezone-aware values are not shifted twice. Ambiguous DST times raise an error.
- `MAX_HR_GAP_SECONDS = 3` limits interpolation across record gaps; invalid rows also block interpolation. `ARTIFACT_WINDOWS` contains reviewed UTC (start, end) intervals. Supported annotations shorter than 10 s may receive cubic-spline repair; other intervals stay missing. Empty annotations mean unreviewed, not artifact-free.
- The input is recorded HR, not a beat-resolved NN series. Its record timestamps do not establish exact physiological timing; upstream smoothing/delay may affect the response.
- AUC tertiles are computed over all candidate bursts before HR exclusions. Your current burst threshold and five-second merge rule are unchanged. No sleep/wake or whole-body classification is inferred.
- Curves show mean +/- event SEM within this recording, not across-subject uncertainty. Thirty-second isolation alone can admit another movement during the late response; the enabled `EXCLUDE_LATE_OVERLAP=True` guard rejects such events through +49 s. This does not guarantee complete physiological recovery between movements.

Outputs remain `hr_events`, `hr_epochs_bpm`, `hr_epochs_pct`, `hr_summary`, and `hr_result["curves"]`. Every candidate retains its QC/exclusion reason. Excluded but HR-complete events have numerical features but do not enter group curves.


In [ ]:
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from IPython.display import display
import burst_hr_response as burst_hr
importlib.reload(burst_hr)

NAIVE_INPUT_TIMEZONE = LOCAL_TZ  # Earlier cells store timezone-naive local wall time.
HR_LIMITS_BPM = (30.0, 200.0)
MAX_HR_GAP_SECONDS = 3.0
ISOLATION_SECONDS = 30.0  # Clear offset-to-onset gaps on both sides.
EXCLUDE_LATE_OVERLAP = True  # Exclude subsequent movement through the HR epoch end.
ARTIFACT_WINDOWS = []  # Reviewed UTC (start, end) pairs; empty means unreviewed.

bbi_hr_input, hr_input_qc = burst_hr.record_hr_to_observations(
    bbi_df, naive_timezone=NAIVE_INPUT_TIMEZONE, hr_limits_bpm=HR_LIMITS_BPM,
)
hr_bursts = bursts_df.copy()
for column in ["start", "end"]:
    hr_bursts[column] = burst_hr.record_times_to_utc(
        hr_bursts[column], naive_timezone=NAIVE_INPUT_TIMEZONE,
    )
hr_recording_times = burst_hr.record_times_to_utc(
    [acc_magnitude_g.index[0], acc_magnitude_g.index[-1]],
    naive_timezone=NAIVE_INPUT_TIMEZONE,
)
display(pd.Series(hr_input_qc, name="Recorded HR input quality"))
print("HR source: bbi_df recorded HR, not interval-derived HR. Sensor origin and upstream processing remain unverified.")


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, constrained_layout=True)
acc_plot_times = burst_hr.record_times_to_utc(
    acc_magnitude_g.index, naive_timezone=NAIVE_INPUT_TIMEZONE,
).tz_convert(LOCAL_TZ)
axes[0].plot(acc_plot_times, acc_magnitude_g * 1000, color="0.5",
             linewidth=0.6, label="Acceleration magnitude")
axes[1].plot(bbi_hr_input.index.tz_convert(LOCAL_TZ), bbi_hr_input["hr_bpm"],
             marker=".", linestyle="-", color="blue", label="FIT-recorded HR from bbi_df")
axes[0].set_ylabel("Magnitude (mg)")
axes[1].set_ylabel("Heart rate (bpm)")
axes[1].set_xlabel(f"Time ({LOCAL_TZ})")
axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %H:%M", tz=LOCAL_TZ))
axes[1].set_xlim(acc_plot_times[0], acc_plot_times[-1])
for ax in axes:
    ax.legend(loc="upper right")
plt.show()

In [ ]:
hr_result = burst_hr.analyze_burst_hr(
    hr_bursts,
    bbi_hr_input,
    recording_start=hr_recording_times[0],
    recording_end=hr_recording_times[-1] + pd.Timedelta(seconds=1 / SAMPLING_RATE_HZ),
    max_gap_s=MAX_HR_GAP_SECONDS,
    artifacts=ARTIFACT_WINDOWS,
    isolation_s=ISOLATION_SECONDS,
    exclude_late_overlap=EXCLUDE_LATE_OVERLAP,
)
hr_result["report"]["hr_source"] = "bbi_df." + hr_input_qc["value_column"]
hr_result["report"]["notes"][0] = "HR uses FIT record timestamps; sensor source, smoothing and physiological delay are unverified."
hr_events = hr_result["events"]
hr_events["exclusion_reason"] = hr_events["exclusion_reason"].str.replace(
    "insufficient_bbi_or_artifact", "insufficient_record_hr_or_artifact", regex=False,
)
hr_epochs_bpm = hr_result["epochs_bpm"]
hr_epochs_pct = hr_result["epochs_pct"]
hr_summary = hr_result["summary"]
q1, q2 = hr_result["report"]["auc_tertile_cutoffs_g_s"]
print(f"AUC cutoffs: {q1:.6g}, {q2:.6g} g*s (all {len(hr_events)} candidate bursts)")
print(f"HR-complete windows: {hr_events.hr_complete.sum()}; included isolated windows: {hr_events.included.sum()}")
late_overlap_n = hr_result["report"]["included_with_other_movement_in_epoch"]
if late_overlap_n:
    print(f"Caution: {late_overlap_n} included windows contain a subsequent movement before the end of the HR epoch. Set EXCLUDE_LATE_OVERLAP=True for sensitivity analysis.")
display(hr_summary)
display(hr_events[[
    "start", "auc_tertile", "AUC", "included", "exclusion_reason",
    "hr_coverage_pct", "baseline_bpm", "hr_peak_increase_pct", "hr_peak_latency_s",
    "hr_post_min_pct", "other_movement_in_epoch",
]].head(20))
display(hr_events.loc[~hr_events.included, "exclusion_reason"].value_counts().rename("Excluded bursts"))

colors = {"Low": "#167D8D", "Medium": "#A6761D", "High": "#B52E55"}
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
for group, color in colors.items():
    curve = hr_result["curves"].query("auc_tertile == @group")
    n = int(hr_summary.loc[group, "included_bursts"])
    axes[0].plot(curve.relative_s, curve.mean_pct, color=color, label=f"{group} AUC (n={n})")
    if n >= 2:
        axes[0].fill_between(curve.relative_s, curve.mean_pct - curve.sem_pct,
                             curve.mean_pct + curve.sem_pct, color=color, alpha=0.15)
    selected = hr_events.loc[hr_events.included & (hr_events.auc_tertile == group)]
    axes[1].scatter(selected.AUC, selected.hr_peak_increase_pct, color=color, alpha=0.75, s=28)
axes[0].axvline(0, color="0.3", linestyle="--")
axes[0].axhline(0, color="0.5", linewidth=0.8)
axes[0].axvspan(burst_hr.RELATIVE_SECONDS[burst_hr.BASELINE][0],
                  burst_hr.RELATIVE_SECONDS[burst_hr.BASELINE][-1] + 1, color="0.7", alpha=0.15)
axes[0].set(xlabel="Seconds from movement onset (approximate HR timing)",
            ylabel="HR change from baseline (%)",
            title="Recorded HR response: mean +/- event SEM")
axes[0].legend()
axes[1].set(xlabel="Burst AUC (g*s)", ylabel="Post-onset HR peak increase (%)",
            title="Movement AUC and HR response")
if hr_events.included.any() and (hr_events.loc[hr_events.included, "AUC"] > 0).all():
    axes[1].set_xscale("log")
    axes[1].set_xlabel("Burst AUC (g*s; logarithmic scale)")
if not hr_events.included.any():
    axes[0].text(0.5, 0.6, "No eligible HR windows", transform=axes[0].transAxes, ha="center")
plt.show()

In [ ]:
# Change BURST_ID to inspect any candidate, including exclusions.
BURST_ID = int(hr_events.index[hr_events.included][0]) if hr_events.included.any() else (
    int(hr_events.index[0]) if len(hr_events) else None
)
if BURST_ID is not None:
    event = hr_events.loc[BURST_ID]
    display(event)
    rel = (bbi_hr_input.index - event.start).total_seconds()
    nearby = (rel >= burst_hr.RELATIVE_SECONDS[0]) & (rel <= burst_hr.RELATIVE_SECONDS[-1])
    fig, ax = plt.subplots(figsize=(11, 4), constrained_layout=True)
    ax.scatter(rel[nearby], bbi_hr_input.loc[nearby, "hr_bpm"], s=18, color="0.5",
               label="FIT-recorded HR from bbi_df (unreviewed)")
    ax.plot(burst_hr.RELATIVE_SECONDS, hr_epochs_bpm.loc[BURST_ID], color="teal",
            marker=".", label="1 Hz HR; gaps left missing")
    ax.axvspan(burst_hr.RELATIVE_SECONDS[burst_hr.BASELINE][0],
               burst_hr.RELATIVE_SECONDS[burst_hr.BASELINE][-1] + 1, color="steelblue", alpha=0.12, label="Baseline")
    ax.axvspan(0, min(int(burst_hr.RELATIVE_SECONDS[-1]), (event.end - event.start).total_seconds()),
               color="gold", alpha=0.25, label="Movement")
    if np.isfinite(event.baseline_bpm):
        ax.axhline(event.baseline_bpm, color="steelblue", linestyle=":")
    ax.axvline(0, color="0.3", linestyle="--")
    status = "included" if event.included else event.exclusion_reason
    ax.set(xlim=(burst_hr.RELATIVE_SECONDS[0], burst_hr.RELATIVE_SECONDS[-1]), xlabel="Seconds from burst onset (record-timed HR)",
           ylabel="HR (bpm)", title=f"Burst {BURST_ID} | {event.auc_tertile} AUC | {status}")
    ax.legend(loc="best")
    plt.show()


## GP pipeline HRV from Garmin BBIs

This section adapts the existing **GP_pipeline** interval workflow to the individual
numbered BBIs delivered by Garmin. It replaces the notebook's previous strict,
uncorrected HRV calculation. The movement-associated recorded-HR analysis above is
unchanged. See [method and provenance](../docs/HRV_PIPELINE.md).

**GP processing rules:**
- For HRV segmentation, retain detected bursts lasting at least **2 s** and add
  **1 s after** each retained burst. Shorter movements remain within GP quiet
  segments. The settings below can retain all bursts if required.
- Skip quiet segments shorter than **1 min**. For segments of **1–5 min**, use the
  entire segment. For longer segments, use **5-min windows**, advancing **1 min**,
  with a final window anchored to the segment end when needed.
- Apply GP's interval-based Kubios classifier once, mask flagged intervals and
  values outside **300–2000 ms**, then interpolate by interval order, including
  filling invalid endpoints from their nearest valid neighbor. Expose every repair.
- Calculate **mean HR, RMSSD, SDNN, and PIP** from cleaned intervals, with at least
  **30 intervals** per window. `pip` is a fraction and `pip_pct` its percentage.
  The GP PIP rule counts zero successive differences: a flat N-interval sequence
  has PIP=(N-2)/N despite zero RMSSD. Quantization can affect this metric.

**Garmin adaptations and quality:**
- Use delivered BBIs directly; do not run the raw-PPG beat detector or invent a
  first interval. Keep each same-callback interval in sequence order.
- Never classify or interpolate across sequence holes or callback gaps over **3 s**.
  A window crossing either barrier is excluded; quiet segments are never joined.
- Use end-exclusive windows and keep both raw and cleaned intervals. Exclude windows
  with unresolved invalid data, excessive edge silence, or sum(cleaned BBI) outside
  **90–110%** of the window duration. This is a consistency check, not physiological
  beat coverage. `HRV_MAX_INTERPOLATED_FRACTION=1.0` matches GP's lack of a repair-rate
  cutoff; lower it for a stricter screen and inspect the repair counts.
- Window placement uses approximate callback-arrival times, not exact beat times.
  GP's validation for its original sensors does not validate Garmin BBIs or their
  source. Quiet segments do not establish sleep or normal-to-normal intervals.
- Window duration varies and windows overlap. Compare like durations and do not
  treat overlapping windows as independent observations.

Outputs: `bbi_intervals` (raw/cleaned values and flags), `hrv_segments`,
`hrv_windows` (including exclusions), and `hrv_result["report"]`. Tables retain UTC;
all time plots display `LOCAL_TZ`.


In [ ]:
import importlib
import matplotlib.dates as mdates
from fit_bbi import decode_snapshots
from burst_hr_response import record_times_to_utc
import gp_hrv
importlib.reload(gp_hrv)

# Defaults from GP_pipeline. To use 60 s windows / 30 s step, set max=60 and step=30.
HRV_MIN_WINDOW_SECONDS = 60.0
HRV_MAX_WINDOW_SECONDS = 300.0
HRV_STEP_SECONDS = 60.0
HRV_MIN_BURST_DURATION_SECONDS = 2.0
HRV_POST_BURST_GUARD_SECONDS = 1.0
HRV_MIN_BEATS = 30
HRV_BBI_LIMITS_MS = (300.0, 2000.0)
HRV_MAX_CALLBACK_GAP_SECONDS = 3.0
HRV_DURATION_FRACTION_LIMITS = (0.90, 1.10)
HRV_MAX_INTERPOLATED_FRACTION = 1.0  # GP imposes no repair-rate cutoff; inspect counts.

bbi_rows, bbi_decode_report = decode_snapshots(
    (message.name, message.get_values())
    for message in messages if message.name in ("record", "session")
)
display(pd.Series(bbi_decode_report, name="Numbered BBI recovery"))
if bbi_decode_report["schema"] is None:
    raise ValueError("This file has no numbered BBI snapshots. Recorded HR cannot replace BBIs for HRV.")

hrv_bursts = bursts_df[["start", "end"]].copy()
for column in ["start", "end"]:
    hrv_bursts[column] = record_times_to_utc(hrv_bursts[column], naive_timezone=LOCAL_TZ)
hrv_recording_bounds = record_times_to_utc(
    [acc_magnitude_g.index[0], acc_magnitude_g.index[-1]], naive_timezone=LOCAL_TZ,
)
hrv_result = gp_hrv.interburst_hrv(
    bbi_rows, hrv_bursts,
    recording_start=max(sleep_start_utc, hrv_recording_bounds[0]),
    recording_end=min(sleep_end_utc, hrv_recording_bounds[1] + pd.Timedelta(seconds=1 / SAMPLING_RATE_HZ)),
    min_window_s=HRV_MIN_WINDOW_SECONDS, max_window_s=HRV_MAX_WINDOW_SECONDS,
    step_s=HRV_STEP_SECONDS, min_burst_duration_s=HRV_MIN_BURST_DURATION_SECONDS,
    post_burst_guard_s=HRV_POST_BURST_GUARD_SECONDS, min_beats=HRV_MIN_BEATS,
    bbi_limits_ms=HRV_BBI_LIMITS_MS, max_callback_gap_s=HRV_MAX_CALLBACK_GAP_SECONDS,
    duration_fraction_limits=HRV_DURATION_FRACTION_LIMITS,
    max_interpolated_fraction=HRV_MAX_INTERPOLATED_FRACTION,
)
bbi_intervals = hrv_result["intervals"]
hrv_segments = hrv_result["segments"]
hrv_windows = hrv_result["windows"]
display(pd.Series(hrv_result["report"], name="GP HRV settings and cleaning"))
print(f"{len(hrv_segments)} GP quiet segments; {len(hrv_windows)} candidate windows.")
print(f"QC included: {hrv_windows.included.sum()}; excluded: {(~hrv_windows.included).sum()}")
display(hrv_segments.assign(
    start_local=hrv_segments.start.dt.tz_convert(LOCAL_TZ),
    end_local=hrv_segments.end.dt.tz_convert(LOCAL_TZ),
)[["start_local", "end_local", "duration_s"]].head(20))
display(hrv_windows.assign(
    start_local=hrv_windows.start.dt.tz_convert(LOCAL_TZ),
    end_local=hrv_windows.end.dt.tz_convert(LOCAL_TZ),
)[["segment_id", "start_local", "end_local", "window_length_s", "mean_hr_bpm",
   "rmssd_ms", "sdnn_ms", "pip_pct", "included", "exclusion_reason", "n_intervals",
   "n_artifacts", "n_interpolated", "interpolated_fraction", "missing_sequences",
   "max_callback_gap_s", "bbi_duration_fraction"]].head(30))
if not hrv_windows.empty:
    display(hrv_windows.loc[~hrv_windows.included, "exclusion_reason"].str.split(";").explode()
            .value_counts().rename("Excluded windows per reason (reasons may overlap)"))
    display(hrv_windows.loc[hrv_windows.included,
                           ["window_length_s", "mean_hr_bpm", "rmssd_ms", "sdnn_ms", "pip_pct"]].describe())
else:
    print(f"No quiet segment is long enough for the {HRV_MIN_WINDOW_SECONDS:g} s minimum window.")

fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True, constrained_layout=True)
for _, group in hrv_windows.groupby("segment_id"):
    times_local = group.center.dt.tz_convert(LOCAL_TZ)
    # Excluded windows stay NaN; separate segments never connect.
    axes[0].plot(times_local, group.rmssd_ms, "o-", ms=3, lw=0.8, color="teal")
    axes[0].plot(times_local, group.sdnn_ms, "o-", ms=3, lw=0.8, color="darkorange")
    axes[1].plot(times_local, group.mean_hr_bpm, "o-", ms=3, lw=0.8, color="purple")
    axes[2].plot(times_local, group.pip_pct, "o-", ms=3, lw=0.8, color="steelblue")
axes[0].plot([], [], color="teal", label="RMSSD")
axes[0].plot([], [], color="darkorange", label="SDNN")
axes[0].legend(loc="upper right")
for burst in hrv_result["blocked_bursts"].itertuples(index=False):
    for ax in axes:
        ax.axvspan(burst.start.tz_convert(LOCAL_TZ), burst.end.tz_convert(LOCAL_TZ),
                   color="firebrick", alpha=0.10)
for _, group in bbi_intervals.loc[bbi_intervals.delivery_run_id >= 0].groupby("delivery_run_id"):
    times_local = group.callback_time_utc_approx.dt.tz_convert(LOCAL_TZ)
    axes[3].plot(times_local, group.bbi_ms, ".", ms=2, color="0.7", alpha=0.5)
    axes[3].plot(times_local, group.bbi_clean_ms, lw=0.7, color="teal")
repaired = bbi_intervals.loc[bbi_intervals.interpolated]
axes[3].scatter(repaired.callback_time_utc_approx.dt.tz_convert(LOCAL_TZ),
                repaired.bbi_clean_ms, s=12, color="darkorange", label="Interpolated")
axes[3].plot([], [], ".", color="0.7", label="Raw BBI")
axes[3].plot([], [], color="teal", label="Cleaned BBI")
axes[3].legend(loc="upper right")
axes[0].set(ylabel="HRV (ms)", title=(
    f"GP quiet-period HRV: {HRV_MIN_WINDOW_SECONDS:g}–{HRV_MAX_WINDOW_SECONDS:g} s windows, "
    f"{HRV_STEP_SECONDS:g} s step + final segment-end window"))
axes[1].set_ylabel("Mean HR (bpm)")
axes[2].set(ylabel="PIP (%)", ylim=(0, 100))
axes[3].set(ylabel="BBI (ms)", xlabel=f"Time ({LOCAL_TZ}; approximate callback arrivals)")
axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%d %H:%M", tz=LOCAL_TZ))
axes[3].set_xlim(sleep_start_utc.tz_convert(LOCAL_TZ), sleep_end_utc.tz_convert(LOCAL_TZ))
if not hrv_windows.included.any():
    axes[0].text(0.5, 0.5, "No windows passed BBI quality screening; inspect the exclusions.",
                 transform=axes[0].transAxes, ha="center", va="center", fontsize=12)
plt.show()


## Optional CSV Export

Set `SAVE_CSV = True` to write parsed signals, GP HRV windows, raw/cleaned BBIs, and analysis settings under `offline_processing/outputs/<fit-file-name>/`. These outputs remain local and are ignored by Git.


In [ ]:
SAVE_CSV = False

if SAVE_CSV:
    import json
    output_dir = PROCESSING / "outputs" / FIT_PATH.stem
    output_dir.mkdir(parents=True, exist_ok=True)
    bbi_df.to_csv(output_dir / "record_bbi_fields.csv", index=False)
    accel_df.to_csv(output_dir / "accelerometer_samples.csv", index=False)
    bbi_intervals.to_csv(output_dir / "gp_bbi_intervals.csv", index=False)
    hrv_segments.to_csv(output_dir / "gp_hrv_segments.csv")
    hrv_windows.to_csv(output_dir / "gp_hrv_windows.csv")
    (output_dir / "gp_hrv_settings.json").write_text(
        json.dumps(hrv_result["report"], indent=2), encoding="utf-8")
    print(f"Wrote analysis outputs to {output_dir.resolve()}")
else:
    print("CSV export skipped. Set SAVE_CSV = True to enable it.")
